# Pathik: which LLM understands our robot commands best?

**What this does:** a person says something to the robot ("take this file to the HOD office and come back").
An LLM must turn that into a task for the robot. This notebook sends **30 test commands** to several LLMs,
checks each answer against the **right answer** (already written for you), and gives every model a score.

**How to use it (about 15 minutes):**
1. Get a free API key at **console.groq.com** → API Keys → Create. Keep it private.
2. Paste it in step 1 below.
3. Menu **Runtime → Run all**.
4. Look at the score table and the list of mistakes at the bottom.

**Your job after the first run:**
- Add **10–15 of your own commands** in step 3 (Hindi-English, polite forms, nicknames, tricky ones).
- Try to **improve the instructions** in step 2 so the best model makes fewer mistakes, then run again.
- Send Arpit: the score table (screenshot), the instructions that worked best, and the commands that still fail.

In [ ]:
# STEP 1: paste your Groq key between the quotes
GROQ_API_KEY = "gsk_paste_your_key_here"

# models to compare (step 1b below lists every model your key can use; copy names from there)
MODELS = ["llama-3.1-8b-instant", "llama-3.3-70b-versatile"]

In [ ]:
# STEP 1b: which models are available right now?
import requests, json, time
r = requests.get("https://api.groq.com/openai/v1/models", headers={"Authorization": f"Bearer {GROQ_API_KEY}"})
print(r.status_code, "" if r.ok else r.text[:300])
if r.ok:
    for m in sorted(x["id"] for x in r.json()["data"]):
        print(" ", m)

## Step 2: our floor, and the instructions the robot gives the LLM

These are the **same instructions the real robot uses**. Change the wording to make the LLM better; keep the
list of output keys the same (the robot reads them).

In [ ]:
import json, time, requests

PLACES = [
    {"name": "HOD office", "aliases": ["hod", "head of department"], "room_contains": ["printer", "files"]},
    {"name": "Lab 3", "aliases": ["computer lab", "lab three"], "room_contains": ["computers", "projector", "whiteboard"]},
    {"name": "Lab 2", "aliases": ["lab two"], "room_contains": ["3D printer", "soldering station", "oscilloscope"]},
    {"name": "Store", "aliases": ["store room"], "room_contains": ["spare batteries", "toolbox", "cables"]},
    {"name": "Faculty room", "aliases": ["staff room"], "room_contains": ["coffee machine", "lockers"]},
    {"name": "Seminar hall", "aliases": ["auditorium"], "room_contains": ["stage", "speakers", "chairs"]},
    {"name": "Water cooler", "aliases": ["water"]},
]

INSTRUCTIONS = (
    "You control an indoor campus robot. Convert the user command into JSON with keys: "
    "intent (one of navigate, return_home, dock (go and charge), stop, status, "
    "explain (user asks why the robot stopped / what went wrong), "
    "find (user asks where something is), confirm (user says ok/done), unknown), "
    "query (for find: the thing asked about), "
    "targets (ordered list of place names, ONLY copied exactly from the provided list), "
    "priority (0 low, 1 normal, 2 high, 3 urgent), "
    "reply (one short friendly sentence to say back), "
    "waits (list of seconds to wait at each target, same order as targets, 0 if none), "
    "then_home (true if the user wants the robot to come back / return afterwards), "
    "confirms (list of true/false per target: true if the robot must wait there until someone presses OK). "
    "A target can be a room; if the user names a thing, pick the room whose room_contains has it. "
    "Never invent places. If a requested place is not in the list, leave it out and say so in reply. "
)

def system_prompt():
    return INSTRUCTIONS + "Known places: " + json.dumps(PLACES)

## Step 3: test commands with the right answers

Each line: what the person says, and what the robot **should** do.
`intent` is always needed; the other fields are checked only if you write them.
- `targets`: places in order · `priority`: 0 low, 1 normal, 3 urgent · `waits`: seconds at each place
- `confirms`: wait for someone to press OK at each place · `then_home`: come back afterwards

**Add your own at the end of the list** (same format).

In [ ]:
TESTS = [
    {'text': 'take this file to the HOD office', 'intent': 'navigate', 'targets': ['HOD office']},
    {'text': 'go to lab three', 'intent': 'navigate', 'targets': ['Lab 3']},
    {'text': 'urgent: take this to the HOD office then lab 3', 'intent': 'navigate', 'targets': ['HOD office', 'Lab 3'], 'priority': 3},
    {'text': 'go to lab 2, wait 2 minutes, then come back', 'intent': 'navigate', 'targets': ['Lab 2'], 'waits': [120], 'then_home': True},
    {'text': 'take these papers to the faculty room and wait for a signature', 'intent': 'navigate', 'targets': ['Faculty room'], 'confirms': [True]},
    {'text': 'take me to the 3D printer', 'intent': 'navigate', 'targets': ['Lab 2']},
    {'text': 'go to the room with the soldering station', 'intent': 'navigate', 'targets': ['Lab 2']},
    {'text': 'where is the oscilloscope', 'intent': 'find'},
    {'text': 'which room has the coffee machine', 'intent': 'find'},
    {'text': 'stop', 'intent': 'stop'},
    {'text': 'stop right now', 'intent': 'stop'},
    {'text': 'come back home', 'intent': 'return_home'},
    {'text': 'go and charge yourself', 'intent': 'dock'},
    {'text': 'why did you stop', 'intent': 'explain'},
    {'text': 'what went wrong', 'intent': 'explain'},
    {'text': 'ok', 'intent': 'confirm'},
    {'text': 'where are you', 'intent': 'status'},
    {'text': 'go to the seminar hall then the store, then return', 'intent': 'navigate', 'targets': ['Seminar hall', 'Store'], 'then_home': True},
    {'text': 'go to the water cooler', 'intent': 'navigate', 'targets': ['Water cooler']},
    {'text': 'bring this to the staff room when you are free', 'intent': 'navigate', 'targets': ['Faculty room'], 'priority': 0},
    {'text': 'go to principl offce', 'intent': 'unknown'},
    {'text': 'go to the canteen', 'intent': 'unknown'},
    {'text': 'hod sir ke cabin mein file de do', 'intent': 'navigate', 'targets': ['HOD office']},
    {'text': 'lab 3 jao aur 5 minute ruko phir wapas aao', 'intent': 'navigate', 'targets': ['Lab 3'], 'waits': [300], 'then_home': True},
    {'text': 'please deliver this to the auditorium', 'intent': 'navigate', 'targets': ['Seminar hall']},
    {'text': 'can you go to the store room and wait there for half a minute', 'intent': 'navigate', 'targets': ['Store'], 'waits': [30]},
    {'text': 'go to lab 2 and wait until someone presses ok, then go to the HOD office', 'intent': 'navigate', 'targets': ['Lab 2', 'HOD office'], 'confirms': [True, False]},
    {'text': 'take this to the room with spare batteries', 'intent': 'navigate', 'targets': ['Store']},
    {'text': 'is anything wrong', 'intent': 'explain'},
    {'text': 'dance for me', 'intent': 'unknown'},
    # ---- add yours below this line ----
    # {"text": "hod sir ko ye file de ke aao", "intent": "navigate", "targets": ["HOD office"], "then_home": True},
]
print(len(TESTS), 'tests')

## Step 4: run every model on every command (just run this)

In [ ]:
import json, time, requests
import pandas as pd

def ask(model, text):
    body = {"model": model, "temperature": 0, "max_tokens": 250, "response_format": {"type": "json_object"},
            "messages": [{"role": "system", "content": system_prompt()}, {"role": "user", "content": text}]}
    t0 = time.time()
    for attempt in range(3):
        r = requests.post("https://api.groq.com/openai/v1/chat/completions", json=body,
                          headers={"Authorization": f"Bearer {GROQ_API_KEY}"}, timeout=30)
        if r.status_code == 429:                  # free tier rate limit: wait and retry
            time.sleep(5 * (attempt + 1)); t0 = time.time(); continue
        break
    ms = (time.time() - t0) * 1000
    if not r.ok:
        return {"error": r.text[:200]}, ms
    try:
        return json.loads(r.json()["choices"][0]["message"]["content"]), ms
    except Exception as e:
        return {"error": f"bad JSON: {e}"}, ms

NAMES = {p["name"] for p in PLACES}

def check(test, got):
    problems = []
    if "error" in got:
        return ["no answer: " + got["error"]]
    if got.get("intent") != test["intent"]:
        problems.append(f"intent: wanted {test['intent']}, got {got.get('intent')}")
    targets = got.get("targets") or []
    invented = [t for t in targets if t not in NAMES]
    if invented:
        problems.append(f"INVENTED places: {invented}")
    if "targets" in test and targets != test["targets"]:
        problems.append(f"places: wanted {test['targets']}, got {targets}")
    if "priority" in test and got.get("priority") != test["priority"]:
        problems.append(f"priority: wanted {test['priority']}, got {got.get('priority')}")
    if "waits" in test:
        w = [round(float(x or 0)) for x in (got.get("waits") or [])]
        if w != test["waits"]:
            problems.append(f"waits: wanted {test['waits']}, got {w}")
    if "confirms" in test and [bool(x) for x in (got.get("confirms") or [])] != test["confirms"]:
        problems.append(f"wait-for-OK: wanted {test['confirms']}, got {got.get('confirms')}")
    if "then_home" in test and bool(got.get("then_home")) != test["then_home"]:
        problems.append(f"come back: wanted {test['then_home']}, got {got.get('then_home')}")
    return problems

rows = []
for model in MODELS:
    print("testing", model, "...")
    for t in TESTS:
        got, ms = ask(model, t["text"])
        probs = check(t, got)
        rows.append({"model": model, "command": t["text"], "correct": not probs, "ms": round(ms),
                     "invented_place": any("INVENTED" in p for p in probs),
                     "problems": "; ".join(probs), "reply": got.get("reply", "")})
        time.sleep(0.3)
df = pd.DataFrame(rows)
print("done")

## Step 5: the score table (screenshot this)

In [ ]:
score = df.groupby("model").agg(correct=("correct", "sum"), tests=("correct", "size"),
                                 median_ms=("ms", "median"), invented_places=("invented_place", "sum"))
score["accuracy_%"] = (100 * score["correct"] / score["tests"]).round(0)
score.sort_values("accuracy_%", ascending=False)

## Step 6: what went wrong (use this to improve the instructions in step 2)

In [ ]:
pd.set_option("display.max_colwidth", 120)
df[~df["correct"]][["model", "command", "problems"]]

## Step 7: save everything
Download `pathik_llm_results.csv` from the folder icon on the left and send it to Arpit, together with:
- a screenshot of the score table,
- your best version of `INSTRUCTIONS` (step 2), if you changed it,
- the commands that still fail.

In [ ]:
df.to_csv("pathik_llm_results.csv", index=False)
print("saved pathik_llm_results.csv")

---
## Part 2 (optional, later): robot ears, i.e. speech to text with Whisper
Record 5–10 commands on your phone, upload them with the folder icon on the left, then run this.
It shows what Whisper heard for each clip. Check the place names.

In [ ]:
!pip -q install openai-whisper
import whisper, glob
for size in ["tiny", "base"]:
    model = whisper.load_model(size)
    print(f"\n=== whisper {size} ===")
    for f in sorted(glob.glob("*.m4a") + glob.glob("*.mp3") + glob.glob("*.wav") + glob.glob("*.ogg")):
        t0 = time.time(); text = model.transcribe(f, language="en")["text"]
        print(f"{f}: {text.strip()}   ({time.time() - t0:.1f} s)")